# RAG evaluation

Шесть шагов: env → данные → цепочка → ответы → RAGAS → сохранение метрик.

In [6]:
"""Ячейка 1 — импорты и переменные окружения.

`load_dotenv()` подтягивает `LLM_API_KEY` из `.env`, без него RAGAS-метрики
(Ячейка 5) не смогут поднять LLM-клиент. `json` и `Path` нужны только в
Ячейке 6 для записи итогового файла с метриками.

KERNEL cwd = notebooks/, поэтому корень репо добавляем в sys.path —
иначе `from app...` в ячейках 3–5 не найдётся.
"""
import json
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")
print("python =", sys.executable)
print("ROOT =", ROOT)
print("app importable:", (ROOT / "app").is_dir())
assert "task-api" in sys.executable, (
    "Выбери kernel «Python (task-api)» — сейчас не тот интерпретатор:\n"
    f"  {sys.executable}"
)

python = /Users/Misha/anaconda3/envs/task-api/bin/python
ROOT = /Users/Misha/Desktop/task-api
app importable: True


In [7]:
"""Ячейка 2 — golden-датасет из 10 эталонных вопросов.

Структура: 7 английских вопросов по 3 темам корпуса (linear_model / tree
/ model_evaluation), 2 русских для проверки мультиязычного retrieval'а,
1 meta-вопрос про `about.md`. Поле `ground_truth_url_keywords` нужно для
Recall@k в Ячейке 4: считаем hit, если URL retrieved-чанка содержит
ожидаемое ключевое слово.
"""
GOLDEN = [
    # --- 7 EN in-corpus вопросов ---
    {
        "question": "How does Ridge regression handle multicollinearity?",
        "ground_truth": "Ridge adds an L2 penalty alpha * sum(w_i^2) to the loss, "
                        "which shrinks correlated coefficients toward each other.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does the alpha parameter control in Ridge?",
        "ground_truth": "Alpha controls regularization strength; larger alpha means "
                        "stronger penalty and smaller coefficients.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What is the difference between Lasso and Ridge?",
        "ground_truth": "Lasso uses L1 penalty which can zero out coefficients (feature "
                        "selection); Ridge uses L2 which shrinks but never zeroes.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "What does min_samples_leaf control in a decision tree?",
        "ground_truth": "min_samples_leaf is the minimum number of samples required to be "
                        "at a leaf node; higher values prevent overfitting by limiting depth.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "When does a decision tree overfit?",
        "ground_truth": "Trees overfit when grown too deep without min_samples_leaf or "
                        "min_samples_split constraints, memorising training noise.",
        "ground_truth_url_keywords": ["tree"],
    },
    {
        "question": "What is the formula for precision?",
        "ground_truth": "precision = TP / (TP + FP). Fraction of positive predictions that "
                        "are actually positive.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    {
        "question": "When is recall more important than precision?",
        "ground_truth": "Recall matters most when missing positives is costly: cancer "
                        "screening, fraud detection, anything where false negatives are "
                        "more harmful than false positives.",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 2 RU вопроса (тест мультиязычного retrieval) ---
    {
        "question": "Что такое L2-регуляризация?",
        "ground_truth": "L2-регуляризация добавляет к функции потерь штраф, "
                        "пропорциональный сумме квадратов коэффициентов модели. "
                        "Используется в Ridge.",
        "ground_truth_url_keywords": ["linear_model"],
    },
    {
        "question": "Что такое F1-мера?",
        "ground_truth": "F1 — гармоническое среднее precision и recall, "
                        "F1 = 2 * precision * recall / (precision + recall).",
        "ground_truth_url_keywords": ["model_evaluation"],
    },
    # --- 1 meta-вопрос (тест about.md) ---
    {
        "question": "Что ты умеешь?",
        "ground_truth": "Отвечаю на вопросы по трём разделам scikit-learn: линейные "
                        "модели, деревья решений, метрики качества.",
        "ground_truth_url_keywords": ["about.md", "local"],
    },
]

In [8]:
"""Ячейка 3 — прямое подключение к RAG-pipeline без HTTP-слоя.

`build_rag_chain()` поднимает retriever (Qdrant + embedder) и LCEL-цепочку
в этом же процессе. Прямой вызов даёт чистые тайминги и доступ к
`retrieved_contexts`, которые понадобятся RAGAS в Ячейке 5.
"""
from app.config import settings
from app.rag.chain import build_rag_chain
chain, retriever = build_rag_chain()

/Users/Misha/anaconda3/envs/task-api/lib/python3.11/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5378.84it/s]


In [9]:
"""Ячейка 4 — Recall@k для retriever'а.

Прогоняем каждый golden-вопрос через retriever, забираем URL'ы топ-k чанков
и считаем hit по keyword-совпадению. Дополнительно копим
`retrieved_contexts` — они уйдут в RAGAS на следующем шаге.
"""
def url_match(returned_urls: list[str], expected_keywords: list[str]) -> bool:
    """Помечает retrieval как hit, если хоть один URL содержит ожидаемое ключевое слово.

    Используется в Recall@k: keyword-matching на уровне модуля sklearn
    (`linear_model` / `tree` / `model_evaluation`) — этого достаточно
    для понимания «не промазал ли retriever мимо темы целиком».
    """
    if not expected_keywords:  # ловушка под OOC-вопросы, если решите добавлять
        return not returned_urls
    return any(
        any(kw.lower() in url.lower() for kw in expected_keywords)
        for url in returned_urls
    )

results = []
for item in GOLDEN:
    docs = retriever.invoke(item["question"])
    urls = [doc.metadata.get("source", "") for doc in docs]
    results.append({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "retrieved_urls": urls,
        "retrieved_contexts": [doc.page_content for doc in docs],
        "retriever_hit": url_match(urls, item["ground_truth_url_keywords"]),
    })

hits = sum(1 for r in results if r["retriever_hit"])
recall_at_k = hits / len(results)
print(f"Retriever Recall@{settings.top_k}: {recall_at_k:.3f}  ({hits}/{len(results)})")

Retriever Recall@4: 0.900  (9/10)


In [10]:
"""Ячейка 5 — generation-метрики через RAGAS.

RAGAS 0.2.x требует Dataset со строго заданными именами полей
(`user_input` / `response` / `retrieved_contexts` / `reference`). LLM
и эмбеддер оборачиваем в `LangchainLLMWrapper` и
`LangchainEmbeddingsWrapper` — без этого `evaluate()` не поймёт, как
через них ходить. Считаем Faithfulness (опора на контекст) и
ResponseRelevancy (релевантность ответа вопросу).
"""
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_huggingface import HuggingFaceEmbeddings

from app.llm import get_llm

# Прогоняем chain и собираем ответы
for r in results:
    r["response"] = chain.invoke(r["question"])

# RAGAS 0.2.x требует именно эти имена полей в Dataset:
# user_input / response / retrieved_contexts / reference
ragas_data = Dataset.from_list([
    {
        "user_input": r["question"],
        "response": r["response"],
        "retrieved_contexts": r["retrieved_contexts"],
        "reference": r["ground_truth"],
    }
    for r in results
])

# LLM и эмбеддер для RAGAS оборачиваются в специальные wrapper'ы
llm = LangchainLLMWrapper(get_llm())
emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-small",
        encode_kwargs={"normalize_embeddings": True},
    )
)

ragas_scores = evaluate(
    dataset=ragas_data,
    metrics=[
        Faithfulness(llm=llm),
        ResponseRelevancy(llm=llm, embeddings=emb),
    ],
)
print(ragas_scores)

Evaluating:  65%|██████▌   | 13/20 [00:29<00:26,  3.72s/it]Exception raised in Job[10]: OutputParserException(Invalid json output: The output string did not satisfy the constraints given in the prompt. Fix the output string and return it. The correct output in JSON format, following the specified schema and ensuring no single quotes are used, is: {"statements": ["The formula for precision is given by the equation precision equals tp divided by the sum of tp and fp.", "tp in the precision formula represents the true positives.", "fp in the precision formula represents the false positives."]}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Evaluating: 100%|██████████| 20/20 [01:31<00:00,  4.60s/it]

{'faithfulness': 0.8970, 'answer_relevancy': 0.9403}


In [11]:
"""Ячейка 6 — сохраняем сводку метрик в JSON.

Берём mean по каждой RAGAS-метрике, добавляем retriever-Recall@k и
метаданные прогона (модель, embedder, top_k). Файл `rag_metrics.json`
цитируется в README + входит в будущую легенду на собесе.
"""
# RAGAS 0.2.x возвращает EvaluationResult — берём .to_pandas() для агрегации
df = ragas_scores.to_pandas()
gen_scores = {
    "faithfulness": float(df["faithfulness"].mean()),
    "answer_relevancy": float(df["answer_relevancy"].mean()),
}

metrics = {
    "n_questions": len(GOLDEN),
    "model": settings.llm_model,
    "embedding_model": settings.embedding_model,
    "top_k": settings.top_k,
    "retriever": {
        f"recall_at_{settings.top_k}": recall_at_k,
        "hits": hits,
    },
    "generation": gen_scores,
}
out = ROOT / "notebooks" / "rag_metrics.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(metrics, indent=2, ensure_ascii=False))
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("saved →", out)

{
  "n_questions": 10,
  "model": "meta-llama/llama-3.3-70b-instruct",
  "embedding_model": "intfloat/multilingual-e5-small",
  "top_k": 4,
  "retriever": {
    "recall_at_4": 0.9,
    "hits": 9
  },
  "generation": {
    "faithfulness": 0.8970238095238096,
    "answer_relevancy": 0.9403443692872703
  }
}
saved → /Users/Misha/Desktop/task-api/notebooks/rag_metrics.json


In [ ]:
import tkinter as tk

def calculate_ctr():
    try:
        impressions = int(entry_impressions.get())
        clicks = int(entry_clicks.get())

        ctr = clicks / impressions * 100

        if ctr < 1:
            rating = "Слабая кампания"
        elif ctr <= 3:
            rating = "Нормальная кампания"
        else:
            rating = "Хорошая кампания"

        result_label.config(
            text=f"CTR: {ctr:.2f}%\nОценка: {rating}"
        )

    except ZeroDivisionError:
        result_label.config(
            text="Ошибка: количество показов не может быть равно 0"
        )

    except ValueError:
        result_label.config(
            text="Ошибка: введите числа"
        )


root = tk.Tk()
root.title("Расчёт CTR")


label_impressions = tk.Label(root, text="Количество показов:")
label_impressions.pack()

entry_impressions = tk.Entry(root)
entry_impressions.pack()


label_clicks = tk.Label(root, text="Количество кликов:")
label_clicks.pack()

entry_clicks = tk.Entry(root)
entry_clicks.pack()


button = tk.Button(
    root,
    text="Рассчитать CTR",
    command=calculate_ctr
)
button.pack()


result_label = tk.Label(root, text="Результат появится здесь")
result_label.pack()


root.mainloop()